# Cross-Domain Evaluation (Zero-Shot)

This notebook implements **Objective 2 / Research Question 1**.

The objective is to evaluate a model trained on **TrashNet** directly on the
**RealWaste** dataset **without any retraining**.

The notebook performs:

- Load a pretrained TrashNet model
- Evaluate on the overlapping RealWaste classes
- Calculate Accuracy
- Calculate Macro F1-score
- Generate Confusion Matrix
- Generate Classification Report
- Measure the Clean-to-Real Accuracy Gap
- Save results as JSON

In [1]:
import json
import os

import torch

from torch.utils.data import DataLoader

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

import config

from dataset import (
    get_clean_transforms,
    get_heavy_transforms,
    make_realwaste_splits,
)

from models import build_model

In [2]:
BACKBONE = "resnet50"

# Available models:
# "resnet50"
# "efficientnet_b0"
# "mobilenet_v3_large"

In [3]:
def run_zero_shot(backbone_name):

    device = config.DEVICE

    checkpoint = os.path.join(
        config.CHECKPOINT_DIR,
        f"{backbone_name}_trashnet_best.pt"
    )

    if not os.path.exists(checkpoint):

        raise FileNotFoundError(
            f"Checkpoint not found:\n{checkpoint}"
        )

    _, eval_tf = get_clean_transforms()

    _, finetune_tf = get_heavy_transforms()

    zero_shot_ds, _, _, _ = make_realwaste_splits(
        eval_tf,
        finetune_tf
    )

    loader = DataLoader(
        zero_shot_ds,
        batch_size=config.BATCH_SIZE,
        shuffle=False,
        num_workers=2
    )

    model, _ = build_model(
        backbone_name,
        len(config.TRASHNET_CLASSES),
        pretrained=False
    )

    model.load_state_dict(
        torch.load(
            checkpoint,
            map_location=device
        )
    )

    model = model.to(device)

    model.eval()

    predictions = []

    labels_all = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)

            outputs = model(images)

            preds = outputs.argmax(dim=1)

            predictions.extend(
                preds.cpu().numpy()
            )

            labels_all.extend(
                labels.numpy()
            )

    accuracy = accuracy_score(
        labels_all,
        predictions
    )

    macro_f1 = f1_score(
        labels_all,
        predictions,
        average="macro"
    )

    cm = confusion_matrix(
        labels_all,
        predictions,
        labels=list(range(len(config.TRASHNET_CLASSES)))
    )

    report = classification_report(
        labels_all,
        predictions,
        labels=list(range(len(config.TRASHNET_CLASSES))),
        target_names=config.TRASHNET_CLASSES,
        output_dict=True,
        zero_division=0
    )

    return accuracy, macro_f1, cm, report

In [4]:
accuracy, macro_f1, confusion, report = run_zero_shot(
    BACKBONE
)

print("Zero-Shot Evaluation Complete")

Zero-Shot Evaluation Complete


In [5]:
print(f"Accuracy : {accuracy:.4f}")

print(f"Macro F1 : {macro_f1:.4f}")

Accuracy : 0.3772
Macro F1 : 0.3512


In [6]:
baseline_path = os.path.join(
    config.RESULTS_DIR,
    f"{BACKBONE}_trashnet_results.json"
)

clean_accuracy = None

if os.path.exists(baseline_path):

    with open(baseline_path) as f:

        baseline = json.load(f)

    clean_accuracy = baseline.get(
        "trashnet_test_accuracy"
    )

if clean_accuracy is not None:

    gap = clean_accuracy - accuracy

    print("TrashNet Accuracy :", clean_accuracy)

    print("Clean → Real Gap :", gap)

TrashNet Accuracy : 0.9421052631578948
Clean → Real Gap : 0.5648847824891383


In [7]:
import pandas as pd

cm_df = pd.DataFrame(
    confusion,
    index=config.TRASHNET_CLASSES,
    columns=config.TRASHNET_CLASSES
)

cm_df

,cardboard,glass,metal,paper,plastic,trash
cardboard,228,5,38,40,33,17
glass,72,112,24,24,80,20
metal,172,34,385,11,16,28
paper,251,1,14,91,15,26
plastic,256,41,131,22,237,54
trash,196,4,53,76,34,30


In [8]:
report_df = pd.DataFrame(report).transpose()

report_df

,precision,recall,f1-score,support
cardboard,0.194043,0.631579,0.296875,361.00000
glass,0.568528,0.337349,0.423440,332.00000
metal,0.596899,0.595975,0.596437,646.00000
paper,0.344697,0.228643,0.274924,398.00000
plastic,0.571084,0.319838,0.410035,741.00000
trash,0.171429,0.076336,0.105634,393.00000
accuracy,0.377220,0.377220,0.377220,0.37722
macro avg,0.407780,0.364953,0.351224,2871.00000
weighted avg,0.443097,0.377220,0.378900,2871.00000


In [9]:
results = {

    "backbone": BACKBONE,

    "realwaste_zero_shot_accuracy": accuracy,

    "realwaste_zero_shot_macro_f1": macro_f1,

    "clean_trashnet_accuracy": clean_accuracy,

    "clean_to_real_gap":
        clean_accuracy - accuracy
        if clean_accuracy is not None
        else None,

    "confusion_matrix":
        confusion.tolist(),

    "classification_report":
        report
}

save_path = os.path.join(
    config.RESULTS_DIR,
    f"{BACKBONE}_cross_domain_results.json"
)

with open(save_path, "w") as f:

    json.dump(
        results,
        f,
        indent=4
    )

print("Results saved to")

print(save_path)

Results saved to
e:\wasteClassificationProject_FIXED\wasteClassificationProject\outputs\resnet50_cross_domain_results.json


In [10]:
results

{'backbone': 'resnet50',
 'realwaste_zero_shot_accuracy': 0.3772204806687565,
 'realwaste_zero_shot_macro_f1': 0.35122420008687216,
 'clean_trashnet_accuracy': 0.9421052631578948,
 'clean_to_real_gap': 0.5648847824891383,
 'confusion_matrix': [[228, 5, 38, 40, 33, 17],
  [72, 112, 24, 24, 80, 20],
  [172, 34, 385, 11, 16, 28],
  [251, 1, 14, 91, 15, 26],
  [256, 41, 131, 22, 237, 54],
  [196, 4, 53, 76, 34, 30]],
 'classification_report': {'cardboard': {'precision': 0.19404255319148936,
   'recall': 0.631578947368421,
   'f1-score': 0.296875,
   'support': 361.0},
  'glass': {'precision': 0.5685279187817259,
   'recall': 0.3373493975903614,
   'f1-score': 0.42344045368620037,
   'support': 332.0},
  'metal': {'precision': 0.5968992248062015,
   'recall': 0.5959752321981424,
   'f1-score': 0.5964368706429125,
   'support': 646.0},
  'paper': {'precision': 0.3446969696969697,
   'recall': 0.228643216080402,
   'f1-score': 0.27492447129909364,
   'support': 398.0},
  'plastic': {'precisio

In [11]:
all_results = {}

for backbone in config.BACKBONES:

    print("="*60)

    print(backbone)

    acc, f1, _, _ = run_zero_shot(backbone)

    all_results[backbone] = {

        "Accuracy": acc,

        "Macro F1": f1
    }

pd.DataFrame(all_results).T

resnet50
efficientnet_b0
mobilenet_v3_large


,Accuracy,Macro F1
resnet50,0.377220,0.351224
efficientnet_b0,0.337513,0.325880
mobilenet_v3_large,0.433647,0.420376
